# 03 - 融合算子：减少内存带宽的关键技术

> **本 Notebook 涵盖内容**
> - 为什么需要算子融合（Memory-bound vs Compute-bound）
> - 分析内存带宽瓶颈
> - 实现 Fused SiLU-And-Mul（与 vLLM 的 SwigluStepAndMul 对应）
> - 实现 Fused Add-RMSNorm（与 vLLM 的 fused_add_rms_norm 对应）
> - 性能对比：融合 vs 非融合
> - Triton autotune 自动调优

**运行示例**：继续使用 SiLU/SwiGLU，但这次关注「融合」带来的性能提升。

## 1. 先感受一下内存带宽的瓶颈

在 Transformer 的 MLP 层中，标准的 SwiGLU 操作是这样的：

```python
# 标准 PyTorch（3 次内存读写）
gate = x[:, :d]        # 读 x，写 gate — 内存操作 1
up = x[:, d:]           # 读 x，写 up — 内存操作 2
gate_silu = F.silu(gate)  # 读 gate，写 gate_silu — 内存操作 3
result = gate_silu * up   # 读 gate_silu + up，写 result — 内存操作 4
```

每次内存操作都需要在 GPU 的 HBM（高带宽内存）和计算单元之间传输数据。这些传输是有限速的！

**关键指标：算术强度（Arithmetic Intensity）**

$$\text{算术强度} = \frac{\text{计算量 (FLOPs)}}{\text{内存访问量 (Bytes)}}$$

**数值示例**（Llama-7B, hidden_size=4096, fp16）：
- SiLU 的计算量：每个元素约 10 FLOPs（sigmoid + mul）
- 每个元素的内存访问：2 bytes (读) + 2 bytes (写) = 4 bytes
- 算术强度 = 10 / 4 = 2.5 FLOPs/Byte

而 A100 GPU 的峰值算力是 312 TFLOPS，内存带宽是 2 TB/s：
- 计算/带宽比 = 312 / 2 = 156 FLOPs/Byte

当算术强度 (2.5) << 计算/带宽比 (156) 时，操作是 **Memory-bound** 的。

**生活类比**：想象一个超级厨师（GPU 计算单元），每秒能切 156 个菜。但食材从冰箱（显存）到切菜板（寄存器）的传送带每秒只能送 1 个菜（2.5 FLOPs/4 Bytes）。厨师大部分时间在等食材！融合算子就是把多种菜一次性从冰箱取出来，一口气处理完，减少传送带的往返次数。

In [ ]:
import torch
import torch.nn.functional as F
import triton
import triton.language as tl
import time

device = torch.device("cuda")

# 模拟 Llama-7B 的尺寸
batch_size = 128
hidden_size = 4096
intermediate_size = 11008
x = torch.randn(batch_size, 2 * intermediate_size, device=device, dtype=torch.float16)

# ---- 非融合版本：多次 kernel launch ----
def unfused_silu_and_mul(x):
    d = x.shape[-1] // 2
    gate = x[:, :d]           # kernel 1: slice
    up = x[:, d:]             # kernel 2: slice
    gate_silu = F.silu(gate)  # kernel 3: silu
    return gate_silu * up     # kernel 4: multiply

# ---- 融合版本：单次 kernel launch ----
def fused_silu_and_mul(x):
    d = x.shape[-1] // 2
    return F.silu(x[:, :d]) * x[:, d:]  # PyTorch 可能部分融合

# 预热
for _ in range(10):
    _ = unfused_silu_and_mul(x)
    _ = fused_silu_and_mul(x)
torch.cuda.synchronize()

# 计时
t_unfused = triton.testing.do_bench(lambda: unfused_silu_and_mul(x))
t_fused_pt = triton.testing.do_bench(lambda: fused_silu_and_mul(x))

print(f"Unfused SiLU+Mul:  {t_unfused:.4f} ms")
print(f"PyTorch fused:     {t_fused_pt:.4f} ms")
print(f"PyTorch speedup:   {t_unfused/t_fused_pt:.2f}x")

## 2. Triton 融合 SiLU-And-Mul

现在让我们用 Triton 将所有操作融合到一个 kernel 中：

In [ ]:
@triton.jit
def fused_silu_and_mul_kernel(
    output_ptr,
    o_stride,
    input_ptr,
    i_stride,
    d: tl.constexpr,        # 输出维度
    BLOCK_SIZE: tl.constexpr,
):
    """
    融合的 SiLU-And-Mul kernel

    一次性完成: gate = x[:d], up = x[d:], output = silu(gate) * up
    数据只从 HBM 读一次，写一次。

    对应 vLLM: activation.py:27-52 (_swiglustep_and_mul_kernel)
    """
    row = tl.program_id(axis=0)
    col_block = tl.program_id(axis=1)

    offsets = col_block * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < d

    # 输入行的起始位置
    input_row = input_ptr + row * i_stride
    output_row = output_ptr + row * o_stride

    # 一次性加载 gate 和 up（它们在内存中是连续的）
    gate = tl.load(input_row + offsets, mask=mask).to(tl.float32)
    up = tl.load(input_row + offsets + d, mask=mask).to(tl.float32)

    # 在寄存器中完成所有计算（不写回中间结果）
    gate_silu = gate * tl.sigmoid(gate)
    result = gate_silu * up

    # 只写一次结果
    tl.store(output_row + offsets, result.to(input_ptr.dtype.element_ty), mask=mask)


def triton_fused_silu_and_mul(x: torch.Tensor) -> torch.Tensor:
    b, n = x.shape
    d = n // 2
    output = torch.empty((b, d), dtype=x.dtype, device=x.device)
    BLOCK_SIZE = 1024

    grid = (b, triton.cdiv(d, BLOCK_SIZE))
    fused_silu_and_mul_kernel[grid](
        output, output.stride(0),
        x, x.stride(0),
        d=d,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return output


# 验证正确性
result = triton_fused_silu_and_mul(x)
expected = unfused_silu_and_mul(x)
max_err = (result - expected).abs().max().item()
print(f"Max error: {max_err:.2e}")
assert max_err < 1e-2
print("Fused SiLU+Mul Triton kernel: PASSED!")

# 性能对比
t_triton = triton.testing.do_bench(lambda: triton_fused_silu_and_mul(x))
print(f"\nPerformance comparison:")
print(f"  Unfused PyTorch: {t_unfused:.4f} ms")
print(f"  Fused PyTorch:   {t_fused_pt:.4f} ms")
print(f"  Fused Triton:    {t_triton:.4f} ms")
print(f"  Triton speedup over unfused: {t_unfused/t_triton:.2f}x")

### 2.1 融合的内存分析

![融合 vs 非融合的内存访问](../diagrams/03-fused-memory.svg)

| 指标 | 非融合 | 融合 |
|------|--------|------|
| HBM 读取 | 3 次 (gate, up, gate_silu) | 1 次 (x) |
| HBM 写入 | 2 次 (gate_silu, result) | 1 次 (result) |
| Kernel launch | 3-4 次 | 1 次 |
| 中间结果内存 | 需要 gate_silu 缓冲区 | 不需要 |

## 3. 融合 Add-RMSNorm：更复杂的融合

RMSNorm (Root Mean Square Normalization) 是 Llama 使用的归一化方式：

$$\text{RMSNorm}(x) = \frac{x}{\sqrt{\frac{1}{n}\sum_{i=1}^{n} x_i^2 + \epsilon}} \cdot w$$

在 Transformer 中，RMSNorm 总是和残差加法一起使用：

```python
# Transformer block 中的标准模式
hidden_states = hidden_states + residual      # 残差加法
hidden_states = rms_norm(hidden_states)        # 归一化
```

vLLM 将这两步融合为一个操作：`fused_add_rms_norm`

**数值示例**（hidden_size=4）：
- 输入 x = [1.0, 2.0, -1.0, 0.5]
- 残差 residual = [0.5, -0.5, 1.0, 0.5]
- 相加后 = [1.5, 1.5, 0.0, 1.0]
- $RMS = \sqrt{\frac{1.5^2 + 1.5^2 + 0^2 + 1.0^2}{4}} = \sqrt{\frac{5.5}{4}} = 1.172$
- 归一化后 = [1.280, 1.280, 0.0, 0.853]

**生活类比**：这就像在流水线上，工人 A 把两个零件焊接在一起（残差加法），然后传给工人 B 做质检和校准（RMSNorm）。融合操作就是让一个工人同时完成焊接和校准——零件不用在两个工位之间传递了。

> **Source**: vllm/model_executor/layers/layernorm.py:56-74

In [ ]:
@triton.jit
def fused_add_rms_norm_kernel(
    x_ptr,           # 输入/输出（in-place）
    residual_ptr,    # 残差（读取后更新）
    weight_ptr,      # 归一化权重
    x_stride,
    r_stride,
    hidden_size,
    eps: tl.constexpr,
    BLOCK_SIZE: tl.constexpr,
):
    """
    融合的 Add + RMSNorm kernel

    同时完成:
    1. new_residual = x + residual
    2. x = RMSNorm(new_residual) * weight

    对应 vLLM: layernorm.py:56-74 (fused_add_rms_norm)

    注意：这个 kernel 是 in-place 的——它修改 x 和 residual。
    """
    row = tl.program_id(axis=0)

    x_row = x_ptr + row * x_stride
    r_row = residual_ptr + row * r_stride

    # 第 1 步：加载 x 和 residual，计算相加结果
    # 同时计算 RMS 所需的平方和
    sum_sq = tl.zeros([1], dtype=tl.float32)

    # 分块处理大的 hidden_size
    for off in range(0, hidden_size, BLOCK_SIZE):
        offsets = off + tl.arange(0, BLOCK_SIZE)
        mask = offsets < hidden_size

        x_val = tl.load(x_row + offsets, mask=mask).to(tl.float32)
        r_val = tl.load(r_row + offsets, mask=mask).to(tl.float32)

        # 融合加法
        added = x_val + r_val

        # 更新 residual（保存加法结果用于下一层）
        tl.store(r_row + offsets, added.to(residual_ptr.dtype.element_ty), mask=mask)

        # 累加平方和
        sum_sq += tl.sum(added * added, axis=0)

    # 第 2 步：计算 RMS
    rms = tl.rsqrt(sum_sq / hidden_size + eps)

    # 第 3 步：归一化并乘以权重
    for off in range(0, hidden_size, BLOCK_SIZE):
        offsets = off + tl.arange(0, BLOCK_SIZE)
        mask = offsets < hidden_size

        # 重新读取 residual（已更新的 x + residual）
        added = tl.load(r_row + offsets, mask=mask).to(tl.float32)
        w = tl.load(weight_ptr + offsets, mask=mask).to(tl.float32)

        # 归一化
        result = added * rms * w
        tl.store(x_row + offsets, result.to(x_ptr.dtype.element_ty), mask=mask)


def triton_fused_add_rms_norm(
    x: torch.Tensor,
    residual: torch.Tensor,
    weight: torch.Tensor,
    eps: float = 1e-6,
) -> tuple[torch.Tensor, torch.Tensor]:
    """
    Python wrapper for fused add + RMSNorm

    Returns: (normalized_x, updated_residual)
    """
    assert x.shape == residual.shape
    b = x.shape[0]
    hidden_size = x.shape[-1]
    BLOCK_SIZE = min(1024, triton.next_power_of_2(hidden_size))

    # In-place kernel
    fused_add_rms_norm_kernel[(b,)](
        x, residual, weight,
        x.stride(0), residual.stride(0),
        hidden_size,
        eps=eps,
        BLOCK_SIZE=BLOCK_SIZE,
    )
    return x, residual


# 验证
hidden_size = 4096
x = torch.randn(128, hidden_size, device=device, dtype=torch.float16)
residual = torch.randn(128, hidden_size, device=device, dtype=torch.float16)
weight = torch.ones(hidden_size, device=device, dtype=torch.float16)

# 保存副本用于对比
x_ref = x.clone()
r_ref = residual.clone()

# PyTorch 参考实现
added_ref = x_ref.float() + r_ref.float()
rms = torch.rsqrt(added_ref.pow(2).mean(dim=-1, keepdim=True) + 1e-6)
expected = (added_ref * rms * weight.float()).half()
expected_residual = added_ref.half()

# Triton 实现
x_out, r_out = triton_fused_add_rms_norm(x, residual, weight, eps=1e-6)

max_err_x = (x_out - expected).abs().max().item()
max_err_r = (r_out - expected_residual).abs().max().item()
print(f"Fused Add+RMSNorm:")
print(f"  Normalized output max error: {max_err_x:.2e}")
print(f"  Residual max error: {max_err_r:.2e}")
assert max_err_x < 1e-2 and max_err_r < 1e-2
print("  PASSED!")

## 4. 性能对比：融合 vs 非融合

In [ ]:
# ---- 非融合 RMSNorm ----
def unfused_add_rms_norm(x, residual, weight, eps=1e-6):
    # 操作 1: 加法
    added = x + residual
    # 操作 2: RMS 计算
    rms = torch.rsqrt(added.float().pow(2).mean(dim=-1, keepdim=True) + eps)
    # 操作 3: 归一化 + 权重
    return (added.float() * rms * weight.float()).to(x.dtype), added

hidden_size = 4096
x = torch.randn(128, hidden_size, device=device, dtype=torch.float16)
residual = torch.randn(128, hidden_size, device=device, dtype=torch.float16)
weight = torch.ones(hidden_size, device=device, dtype=torch.float16)

# 性能测试
def bench_unfused():
    x_c, r_c = x.clone(), residual.clone()
    return unfused_add_rms_norm(x_c, r_c, weight)

def bench_triton():
    x_c, r_c = x.clone(), residual.clone()
    return triton_fused_add_rms_norm(x_c, r_c, weight)

t_unfused = triton.testing.do_bench(bench_unfused)
t_triton = triton.testing.do_bench(bench_triton)

print(f"Fused Add+RMSNorm Performance:")
print(f"  Unfused PyTorch:  {t_unfused:.4f} ms")
print(f"  Fused Triton:     {t_triton:.4f} ms")
print(f"  Speedup:          {t_unfused/t_triton:.2f}x")

# 理论分析
bytes_unfused = 128 * 4096 * 2 * 5  # 5 次 HBM 访问
bytes_fused = 128 * 4096 * 2 * 3    # 3 次 HBM 访问（读 x, residual, weight, 写 x, residual）
print(f"\nTheoretical memory traffic:")
print(f"  Unfused: {bytes_unfused / 1024**2:.1f} MB")
print(f"  Fused:   {bytes_fused / 1024**2:.1f} MB")
print(f"  Reduction: {(1 - bytes_fused/bytes_unfused)*100:.0f}%")

## 5. Triton Autotune: 自动选择最佳配置

vLLM 的 Mamba kernel 使用了 Triton 的 autotune 功能来自动搜索最佳的 BLOCK_SIZE 和 num_warps：

```python
# vllm/model_executor/layers/mamba/ops/ssd_chunk_scan.py:17-30
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_M": 32, "BLOCK_N": 32}, num_stages=1, num_warps=4),
        triton.Config({"BLOCK_M": 64, "BLOCK_N": 64}, num_stages=1, num_warps=4),
        triton.Config({"BLOCK_M": 128, "BLOCK_N": 128}, num_stages=1, num_warps=4),
        # ... 15+ 种配置
    ],
    key=["chunk_size", "hdim", "dstate"],
)
@triton.jit
def _chunk_scan_fwd_kernel(...):
    ...
```

让我们给我们的 SiLU+Mul kernel 也加上 autotune：

In [ ]:
@triton.autotune(
    configs=[
        triton.Config({"BLOCK_SIZE": 256}, num_warps=2),
        triton.Config({"BLOCK_SIZE": 512}, num_warps=4),
        triton.Config({"BLOCK_SIZE": 1024}, num_warps=4),
        triton.Config({"BLOCK_SIZE": 2048}, num_warps=8),
    ],
    key=["d"],  # 当 d 变化时重新搜索最佳配置
)
@triton.jit
def autotuned_silu_and_mul_kernel(
    output_ptr, o_stride,
    input_ptr, i_stride,
    d,
    BLOCK_SIZE: tl.constexpr,
):
    row = tl.program_id(axis=0)
    col_block = tl.program_id(axis=1)

    offsets = col_block * BLOCK_SIZE + tl.arange(0, BLOCK_SIZE)
    mask = offsets < d

    input_row = input_ptr + row * i_stride
    output_row = output_ptr + row * o_stride

    gate = tl.load(input_row + offsets, mask=mask).to(tl.float32)
    up = tl.load(input_row + offsets + d, mask=mask).to(tl.float32)

    gate_silu = gate * tl.sigmoid(gate)
    result = gate_silu * up

    tl.store(output_row + offsets, result.to(input_ptr.dtype.element_ty), mask=mask)


def triton_autotuned_silu_and_mul(x: torch.Tensor) -> torch.Tensor:
    b, n = x.shape
    d = n // 2
    output = torch.empty((b, d), dtype=x.dtype, device=x.device)

    grid = lambda meta: (b, triton.cdiv(d, meta["BLOCK_SIZE"]))
    autotuned_silu_and_mul_kernel[grid](
        output, output.stride(0),
        x, x.stride(0),
        d,
    )
    return output


# 测试各种大小
for intermediate_size in [1024, 4096, 11008, 16384]:
    x = torch.randn(128, 2 * intermediate_size, device=device, dtype=torch.float16)
    result = triton_autotuned_silu_and_mul(x)
    expected = F.silu(x[:, :intermediate_size]) * x[:, intermediate_size:]
    max_err = (result - expected).abs().max().item()

    t = triton.testing.do_bench(lambda: triton_autotuned_silu_and_mul(x))
    print(f"d={intermediate_size:>6d} | Time: {t:.4f} ms | Max error: {max_err:.2e}")

### 5.1 Autotune 的工作原理

![Triton Autotune 工作原理](../diagrams/03-autotune.svg)

关键参数：
- `key=["d"]`：当这些参数变化时，autotune 重新搜索
- `num_warps`：GPU warp 的数量（每个 warp 32 个线程）
- `num_stages`：pipeline 的阶段数（用于隐藏内存延迟）
- `Config`：一组要尝试的配置

## 6. 将融合算子注册为 CustomOp

In [ ]:
# 重用之前的 SimpleCustomOp 基础设施
# 这里直接用 nn.Module 模拟完整流程

class FusedSiluAndMul(torch.nn.Module):
    """
    完整的融合 SiLU+Mul 算子

    设计模式参考 vLLM 的 SwigluStepAndMul (activation.py:341-375)
    """
    name = "fused_silu_and_mul"

    def __init__(self, limit: float = None):
        super().__init__()
        self.limit = limit  # 可选的 clamping 限制

    def forward_native(self, x: torch.Tensor) -> torch.Tensor:
        """PyTorch 原生实现"""
        d = x.shape[-1] // 2
        gate = F.silu(x[..., :d])
        up = x[..., d:]
        if self.limit is not None:
            gate = gate.clamp(max=self.limit)
            up = up.clamp(min=-self.limit, max=self.limit)
        return gate * up

    def forward_cuda(self, x: torch.Tensor) -> torch.Tensor:
        """Triton 融合实现"""
        if x.ndim != 2:
            orig_shape = x.shape
            x = x.view(-1, x.shape[-1])
            result = triton_autotuned_silu_and_mul(x)
            return result.view(orig_shape[:-1] + (result.shape[-1],))
        return triton_autotuned_silu_and_mul(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        if x.is_cuda:
            return self.forward_cuda(x)
        return self.forward_native(x)


# 模型中使用
class DemoMLP(torch.nn.Module):
    def __init__(self, hidden_size=256, intermediate_size=512):
        super().__init__()
        self.gate_up_proj = torch.nn.Linear(hidden_size, 2 * intermediate_size, bias=False)
        self.down_proj = torch.nn.Linear(intermediate_size, hidden_size, bias=False)
        self.act_fn = FusedSiluAndMul()

    def forward(self, x):
        x = self.gate_up_proj(x)
        x = self.act_fn(x)
        x = self.down_proj(x)
        return x


model = DemoMLP().half().to(device)
x = torch.randn(4, 256, device=device, dtype=torch.float16)
output = model(x)
print(f"DemoMLP with fused activation:")
print(f"  Input: {x.shape}")
print(f"  Output: {output.shape}")
print(f"  Activation: {model.act_fn.name}")

# 梯度检查
loss = output.sum()
loss.backward()
print(f"  Backward pass: PASSED!")

## 7. vLLM 中的其他融合模式

vLLM 中常见的融合模式总结：

| 融合算子 | 组合的操作 | 位置 | 性能收益 |
|---------|----------|------|---------|
| `fused_add_rms_norm` | Add + RMSNorm | layernorm.py:56-74 | 减少 1 次 HBM 读写 |
| `swiglustep_and_mul` | SiLU + Clamp + Mul | activation.py:26-74 | 减少 2 次 HBM 读写 |
| `fused_moe_kernel` | Routing + MatMul + Act | fused_moe.py | 减少 3+ 次 HBM 读写 |
| `fused_moe_lora` | LoRA + MoE Routing | fused_moe_lora_op.py | 减少 4+ 次 HBM 读写 |
| Prefill Attention | QK + Softmax + V | triton_prefill_attention.py | FlashAttention 模式 |

融合的一般原则：
1. **元素级操作最适合融合**：它们是纯 memory-bound 的
2. **规约操作（如 RMS 的 mean）需要特殊处理**：需要跨 block 同步
3. **矩阵乘法不适合与元素级操作融合**：它们是 compute-bound 的，自身已经足够快

## 8. 本章小结

| 概念 | 说明 |
|------|------|
| Memory-bound | 性能受限于内存带宽而非计算能力 |
| 算术强度 | FLOPs / Bytes，低于 GPU 阈值则为 memory-bound |
| 算子融合 | 多个操作合并到一个 kernel，减少 HBM 访问 |
| In-place 操作 | 直接修改输入张量，节省内存分配 |
| @triton.autotune | 自动搜索最佳 BLOCK_SIZE, num_warps 等 |
| key 参数 | 当哪些参数变化时触发重新搜索 |

## 源码映射表

| 本教程实现 | vLLM 原始源码 | 章节 |
|-----------|-------------|------|
| `fused_silu_and_mul_kernel` | `activation.py:27-52 (_swiglustep_and_mul_kernel)` | 2 |
| `triton_fused_add_rms_norm` | `layernorm.py:56-74 (fused_add_rms_norm)` | 3 |
| `autotuned_silu_and_mul_kernel` | `mamba/ops/ssd_chunk_scan.py:17-30 (autotune)` | 5 |
| `FusedSiluAndMul` | `activation.py:341-375 (SwigluStepAndMul)` | 6 |

## 下一步

在 [04-replace-model-ops.ipynb](04-replace-model-ops.ipynb) 中，我们将学习如何使用 vLLM 的 OOT 机制替换 Llama 模型中使用的算子实现。